# Kübik Artin $L$-fonksiyonu ve eta ölçeklerinin değiştirilmesi

Bu defter, $t^3-2$ polinomuyla ilişkili Artin $L$-fonksiyonunun bir deformasyonuna dair karşı örneği yeniden üretir. İlk kod hücresi aralık aritmetiğiyle karşı örneği doğrular. Sonraki iki bölüm reel olmayan sıfırların konumunu ve yakındaki yerel çift sıfırı doğrular. Asal sayımı ayrı bir deneydir. Son hücre sonuçları ZIP olarak indirir. Colab'da CPU yeterlidir.

## Soru ve sonuç

Dedekind eta fonksiyonunu kullanarak

$$
W_a(u)=e^{u/2}\eta(i e^{u+a})\eta(i e^{u-a}),\qquad
H_a(z)=\int_0^\infty W_a(u)\cos(zu)\,du
$$

tanımlayalım. $a_0=\frac12\log 3$ için $H_{a_0}$, $t^3-2$'nin Artin $L$-fonksiyonunun tamamlanmış haliyle ilişkilidir. Ailenin $a\ge a_0$ için yalnızca reel sıfırlara sahip olması, bu $L$-fonksiyonunu incelemek için bir yol verebilirdi. **Bu evrensel önerme yanlıştır.** $a=23/16$ için $H_a$'nın en az bir reel olmayan sıfırı vardır. $a=a_0$ için sıfırların konumu bu hesapla belirlenmez.

### Neden negatif işaret yeterli?

Eta ters çevirme eşitliğinden $W_a(-u)=W_a(u)$ elde edilir. $f(z)=\eta(6z)\eta(18z)$ ilgili modüler formdur. Mellin integraline $t=e^u/\sqrt{108}$ konunca, $s=\tfrac12+iz$ için

$$2H_{a_0}(z)=\left(\frac{\sqrt{108}}{2\pi}\right)^s\Gamma(s)L(s,\rho).$$

$W_{23/16}(u)$, $u\ge6$ için $e^{u/2}\exp(-e^u/2)$ ile sınırlı. Bu hem $H_a$'nın tam olduğunu hem de büyümesinin $\log\max_{|z|=r}|H_a(z)|=O(r\log r)$ olduğunu gösterir. Örneğin kuyrukta $v=e^u$ dönüşümüyle bir Gamma integrali üst sınır verir. Dolayısıyla tam fonksiyonun mertebesi en çok 1'dir. Bu durumda bütün sıfırların reel olması klasik Laguerre eşitsizliğini gerektirir.

$$[H_a'(x)]^2-H_a(x)H_a''(x)\ge0\qquad(x\in\mathbb R).$$

Kod, $a=23/16$ ve $x=74/5$ için bu ifadenin yaklaşık $-1{,}74472\cdot10^{-20}$ olduğunu ve hesaplanan aralığın üst ucunun da negatif kaldığını doğrular.

[Modüler form ve eta çarpımı](https://www.lmfdb.org/ModularForm/GL2/Q/holomorphic/108/1/c/a/) · [Eta çarpımı ve dönüşümleri](https://dlmf.nist.gov/27.14) · [Klasik Laguerre eşitsizliği](https://www.numdam.org/item/AMBP_2005__12_2_331_0/)

### Hata sınırları

1. Eta çarpımı reel eksendeki düğümlerde $y\ge1$ olacak biçimde ters çevrilir. $q=e^{-2\pi y}$ için ilk 14 çarpandan sonraki kuyruk $[1-q^{15}/(1-q),1]$ içindedir. Temel fonksiyonlar aralık aritmetiğiyle değerlendirilir.
2. $[0,6]$, uzunluğu $h=1/32$ olan parçalara ayrılır. Her parçada 24. derece interpolasyon polinomunun integrali tam rasyonel ağırlıklarla hesaplanır. İntegrallerin her biri için, $r=1/4$ yarıçaplı Cauchy disklerinde $M=10^6$ üst sınırı geçerlidir. Gerçekten, bu disklerde $\operatorname{Re}(e^{z\pm a})>1/8$ olduğundan her eta çarpanının mutlak değeri $e$'den küçük. Diğer çarpanlar birlikte $24\cdot40\cdot41$ ile sınırlı. Cauchy ve interpolasyon hata formülleri üç integralin her biri için toplam $6M(h/r)^{25}$ hata verir.
3. $u\ge6$ için, $j=0,1,2$ olmak üzere $u^j W_a(u)\le e^{3u/2}\exp(-e^u/2)$. $v=e^u$ dönüşümü kuyruk için $\int_{400}^\infty v e^{-v/2}\,dv=804e^{-200}<10^{-80}$ üst sınırını verir. Her integral için bu hata da aralığa eklenir.

Aralık başına toplam analitik hata üst sınırı $1{,}59\cdot10^{-16}$'dan küçüktür. Sonucu dışa genişletilmiş değerlerle okuyabiliriz. Laguerre ifadesi $[-1{,}744730,-1{,}744713]\cdot10^{-20}$ içindedir.


In [ ]:
from fractions import Fraction

from mpmath import iv


iv.dps = 45
scale = Fraction(23, 16)
frequency = Fraction(74, 5)
resolution = 32
degree = 24
end = 6


def box(rational):
    rational = Fraction(rational)
    return iv.mpf(rational.numerator) / rational.denominator


def weights_for_interpolation(degree):
    weights = []
    for j in range(degree + 1):
        coefficients = [1]
        denominator = 1
        for k in range(degree + 1):
            if j == k:
                continue
            expanded = [0] * (len(coefficients) + 1)
            for power, coefficient in enumerate(coefficients):
                expanded[power] -= k * coefficient
                expanded[power + 1] += coefficient
            coefficients = expanded
            denominator *= j - k
        weights.append(sum(Fraction(c * degree**p, (p + 1) * denominator)
                           for p, c in enumerate(coefficients)))
    assert sum(weights) == 1
    return weights


def eta_on_axis(log_y):
    y = iv.exp(box(abs(log_y)))
    q = iv.exp(-2 * iv.pi * y)
    product = iv.mpf(1)
    for n in range(1, 15):
        product *= 1 - q**n
    omitted = q**15 / (1 - q)
    product *= 1 - iv.mpf([0, 1]) * omitted
    return product * iv.exp(-iv.pi * y / 12 + box(max(-log_y, 0) / 2))


def certify():
    weights = weights_for_interpolation(degree)
    segments = end * resolution
    node_count = segments * degree + 1
    node_weights = [Fraction(0)] * node_count
    for start in range(0, node_count - 1, degree):
        for j, weight in enumerate(weights):
            node_weights[start + j] += weight / resolution

    values = [iv.mpf(0), iv.mpf(0), iv.mpf(0)]
    for i, weight in enumerate(node_weights):
        u = Fraction(i, resolution * degree)
        u_iv = box(u)
        w = box(weight) * iv.exp(u_iv / 2) * eta_on_axis(u + scale) * eta_on_axis(u - scale)
        angle = box(frequency * u)
        c = iv.cos(angle)
        values[0] += w * c
        values[1] -= w * u_iv * iv.sin(angle)
        values[2] -= w * u_iv**2 * c

    # Analytic error bounds for each integral, independent of the interval arithmetic.
    radius = Fraction(1, 4)
    sup_bound = 10**6
    error = end * sup_bound * (1 / (resolution * radius))**(degree + 1) + Fraction(1, 10**80)
    values = [value + iv.mpf([-1, 1]) * box(error) for value in values]
    h, dh, ddh = values
    laguerre = dh**2 - h * ddh
    assert laguerre.b < 0
    return values, laguerre, error


values, laguerre, error = certify()
print("H, H′ ve H″ için doğrulanmış aralıklar")
for value in values:
    print(value)
print("Laguerre ifadesinin aralığı")
print(laguerre)
print("İntegral başına analitik hata üst sınırı", float(error))


## Reel olmayan sıfırların konumu

Şu sabitleri seçelim.

$$a=\frac{23}{16},\qquad c=14.67872714035986218218+0.31107587891741700810i,\qquad r=10^{-12}.$$

Bu hücre $|z-c|<r$ diskinde tam bir sıfır olduğunu Rouché teoremiyle doğrular. Bir önceki kod hücresindeki `box`, `weights_for_interpolation` ve `eta_on_axis` işlevlerini kullanır. $H_a$ gerçel simetriye sahip, bu yüzden eşlenik diskte de tam bir sıfır var. Sayım çoklukla yapıldığı için iki sıfır da basittir.

$H_a(c)$ ve $\operatorname{Re}H_a'(c)$ aralıklarla hesaplanır. Her $|\operatorname{Im}z|\le1$ için $|H_a''(z)|<20$. Gerçekten $W_a(u)\le e^{u/2}\exp(-e^u/2)$ ve $u^2\le e^u$ ile $v=e^u$ dönüşümü üst sınırı $\int_0^{\infty}(1+v^2)e^{-v/2}\,dv=18$ verir. Diskin sınırında $H_a(z)$ ile $H_a'(c)(z-c)$ arasındaki fark böylece $|H_a(c)|+10r^2$ ile sınırlı. Kod bu sınırın $r|\operatorname{Re}H_a'(c)|$ değerinden kesinlikle küçük kaldığını denetler.

İntegraller yine $[0,6]$ üzerinde $1/32$ adımla, bu kez 40. derece interpolasyonla hesaplanır. İntegrandların $u$ değişkenindeki $1/4$ yarıçaplı disklerinde eta çarpanlarının mutlak değerleri $e$ ile sınırlıdır. $|u|<7$, $|e^{u/2}|<24$ ve sinüs veya kosinüs ile hiperbolik çarpanların birleşik büyüklüğü $e^6$ altındadır. Bu yüzden Cauchy hesabında $10^8$ üst sınırı güvenlidir. $u>6$ kuyruğu $\int_{400}^{\infty}v^2 e^{-v/2}\,dv<10^{-80}$ ile sınırlı. Kalan hesap rasyonel ağırlıklar ve aralık aritmetiğiyle yürütülür.


In [ ]:
def certify_nonreal_zero():
    """İlk doğrulama hücresindeki eta ve rasyonel ağırlık yardımcılarını kullanır."""
    iv.dps = 60
    scale = Fraction(23, 16)
    real_text = '14.67872714035986218218'
    imag_text = '0.31107587891741700810'
    real_center, imag_center = Fraction(real_text), Fraction(imag_text)
    radius = Fraction(1, 10**12)
    resolution, degree, end = 32, 40, 6
    weights = weights_for_interpolation(degree)
    count = end * resolution * degree + 1
    node_weights = [Fraction(0)] * count
    for start in range(0, count - 1, degree):
        for i, weight in enumerate(weights):
            node_weights[start + i] += weight / resolution

    real_value, imag_value, real_slope = [iv.mpf(0) for _ in range(3)]
    for i, weight in enumerate(node_weights):
        u = Fraction(i, resolution * degree)
        u_iv = box(u)
        w = box(weight) * iv.exp(u_iv / 2) * eta_on_axis(u + scale) * eta_on_axis(u - scale)
        angle = box(real_center * u)
        exp_beta = iv.exp(box(imag_center * u))
        cosh = (exp_beta + 1 / exp_beta) / 2
        sinh = (exp_beta - 1 / exp_beta) / 2
        real_value += w * iv.cos(angle) * cosh
        imag_value -= w * iv.sin(angle) * sinh
        real_slope -= w * u_iv * iv.sin(angle) * cosh

    # Complex-u Cauchy bound for these integrands and explicit u>6 tail.
    error = end * 10**8 * Fraction(1, 8)**(degree + 1) + Fraction(1, 10**80)
    uncertainty = iv.mpf([-1, 1]) * box(error)
    real_value += uncertainty
    imag_value += uncertainty
    real_slope += uncertainty
    assert real_slope.a > 0
    # On the disk |H''| < 20. Rouché compares H with H'(center)(z-center).
    remainder = abs(real_value) + abs(imag_value) + box(10 * radius**2)
    linear = real_slope * box(radius)
    assert remainder.b < linear.a
    assert imag_center > radius
    print("Üst yarı düzlemdeki diskin merkezi", f"{real_text} + {imag_text}i")
    print("Diskin rasyonel yarıçapı", radius)
    print("Taylor kalanının üst sınırı", remainder.b)
    print("Doğrusal terimin alt sınırı", linear.a)
    print("Her eşlenik diskte tam bir basit sıfır var.")
    return {
        "scale": scale,
        "positive_imaginary_center": {"real": real_text, "imaginary": imag_text},
        "radius": radius,
        "zeros_per_disk_counting_multiplicity": 1,
        "H_at_center": {"real": real_value, "imaginary": imag_value},
        "real_H_prime_at_center": real_slope,
        "rouche_remainder_upper_bound": remainder.b,
        "rouche_linear_lower_bound": linear.a,
        "analytic_error_bound_per_integral": error,
        "interval_precision_digits": iv.dps,
    }


nonreal_certificate = certify_nonreal_zero()


## Yakındaki çift sıfır ve yerel geçiş

Ölçek $a_*$ yaklaşık $1.45200626937306544414$ ve frekans $x_*$ yaklaşık $14.83804011307100864780$ olduğunda $H_{a_*}(x_*)=0$ ve $\partial_x H_{a_*}(x_*)=0$. Aşağıdaki hücre her iki koordinatı rasyonel merkez etrafında yarıçapı $10^{-19}$ olan bir kutuya hapseder ve kutudaki çözümün tek olduğunu doğrular.

### Daralma yoluyla doğrulama

$F(a,x)=(H_a(x),\partial_xH_a(x))$ ve $J=DF$ olsun. Merkez $c$ için $F(c)$ ve $J(c)$ aralıklarla çevrelenir. Koddaki $M$, $J(c)$'nin yaklaşık tersi olarak alınan tersinir bir rasyonel matristir. Böylece $T(y)=y-MF(y)$ sabit noktaları tam olarak $F(y)=0$ çözümleridir.

$a\in[1.4,1.5]$ ve $x\in[14.5,15.5]$ için her iki değişkende yarıçapı $1/4$ olan kompleks polidisklerde $|H_a(x)|<24$. Gerçekten, $u\ge0$ için eta çarpımının içindeki $q$ değerlerinin mutlak değerleri $1/2$ altındadır. Bu yüzden mutlak değerleri alınmış eta çarpımları $e^2\exp(-e^u/3)$ ile sınırlı. Üstel ve kosinüs çarpanları bunu $e^2 e^{3u/4}\exp(-e^u/3)$ ile sınırlar ve integral 24'ten küçüktür. Cauchy eşitsizliği mertebesi en çok üç olan karma türevlerin her birine $24\cdot6\cdot4^3=9216$ üst sınırını verir. Merkez etrafında yarıçap $r$ boyunca $\|J(y)-J(c)\|_\infty\le 50000r$ kullanılır. Hesap $\|T(c)-c\|_\infty<8.6\cdot10^{-28}$ ve $\sup\|DT\|_\infty<1.28\cdot10^{-5}$ verir. Bu iki sınır, $T$'nin kapalı kutuyu kendi içine götüren bir daralma olduğunu gösterir. Banach teoremi tek sabit nokta verir.

### İntegrallerin hata sınırları

Hesap, $H$, $H_x$, $H_a$, $H_{ax}$ ve $H_{xx}$ integrandları için $[0,6]$ aralığında $1/32$ adım ve 48. derece interpolasyon kullanır. $u$ ve $a$ diskleri yarıçapı $1/4$ olduğunda $\operatorname{Re}(e^{u\pm a})>1/10$. Dolayısıyla $|q|<e^{-\pi/5}$ ve her eta çarpanı $e^{6/5}$ ile sınırlı. $u$ kuvvetleri, trigonometrik fonksiyonlar ve ölçek türevi için Cauchy'nin verdiği 4 çarpanı dahil edilince tüm integrandlarda $10^7$ üst sınırı geçerlidir. Toplam interpolasyon hatası her integral için $6\cdot10^7/8^{49}$ değerini aşmaz.

Ölçek türevi için $\ell(v)=\log\eta(ie^v)$ alınır. $v\ge0$ ve $q=\exp(-2\pi e^v)$ için

$$\ell'(v)=\pi e^v\left(2\sum_{n\ge1}\frac{nq^n}{1-q^n}-\frac{1}{12}\right),\qquad \ell'(v)=-\tfrac12-\ell'(-v)\quad(v<0).$$

Seri $N=24$ sonrasında kesilir. Kalan pozitif seri $q^{N+1}(N+1-Nq)/(1-q)^3$ ile sınırlı. Eta çarpımının kuyruğu için önceki hücreyle aynı eşitsizlik kullanılır. $u>6$ kuyruğu, ölçek türevi için Cauchy'nin 4 çarpanı da dahil, $32\int_{400}^\infty v e^{-v/2}\,dv<10^{-80}$ üst sınırına sahiptir.

### İki tarafta ne oluyor?

Çift sıfır kutusunda $\partial_aH<0$ ve $\partial_x^2H>0$. Bu yüzden $\partial_xH=0$ yakın çevrede tek bir analitik kritik nokta $x_c(a)$ tanımlar. $h(a)=H_a(x_c(a))$ fonksiyonunun $a_*$'daki türevi negatiftir. Taylor açılımında

$$H_a(x)=h(a)+(x-x_c(a))^2Q(a,x),\qquad Q(a_*,x_*)=\tfrac12\partial_x^2 H_{a_*}(x_*)>0.$$

Uygun yerel koordinat $y=(x-x_c(a))\sqrt{Q(a,x)/Q(a_*,x_*)}$ ile bu denklem $y^2=-h(a)/Q(a_*,x_*)$ olur. Bu nedenle **$a_*$'ın hemen altında bir eşlenik reel olmayan çift, hemen üstünde iki reel sıfır** vardır. Bu sonuç çift sıfırın yakın çevresi için geçerlidir. Ayrı olarak doğrulanan $a=23/16$ sıfırlarıyla aradaki dalın tamamı burada izlenmez.


In [ ]:
from fractions import Fraction

from mpmath import iv


def certify_collision(scale_text, frequency_text, inverse_text):
    """Verilen rasyonel merkezin yakınında H=H_x=0 çözümünü doğrular."""
    iv.dps = 70
    scale = Fraction(scale_text)
    frequency = Fraction(frequency_text)
    resolution = 32
    degree = 48
    end = 6
    terms = 24
    radius = Fraction(1, 10**19)
    assert Fraction('1.4') <= scale - radius <= scale + radius <= Fraction('1.5')
    assert Fraction('14.5') <= frequency - radius <= frequency + radius <= Fraction('15.5')

    def box(number):
        number = Fraction(number)
        return iv.mpf(number.numerator) / number.denominator

    def norm(vector):
        return max(abs(item).b for item in vector)

    def matrix_norm(matrix):
        return max(sum(abs(item) for item in row).b for row in matrix)

    def interpolation_weights():
        weights = []
        for j in range(degree + 1):
            coefficients = [1]
            divisor = 1
            for k in range(degree + 1):
                if j != k:
                    expanded = [0] * (len(coefficients) + 1)
                    for power, coefficient in enumerate(coefficients):
                        expanded[power] -= k * coefficient
                        expanded[power + 1] += coefficient
                    coefficients = expanded
                    divisor *= j - k
            weights.append(sum(Fraction(c * degree**p, (p + 1) * divisor)
                               for p, c in enumerate(coefficients)))
        # Check polynomial exactness over [0,1] in exact rational arithmetic.
        for power in range(degree + 1):
            assert sum(w * Fraction(j, degree)**power for j, w in enumerate(weights)) == Fraction(1, power + 1)
        return weights

    unit_interval = iv.mpf([0, 1])
    symmetric_interval = iv.mpf([-1, 1])

    def eta_and_log_derivative(log_y):
        y = iv.exp(box(abs(log_y)))
        q = iv.exp(-2 * iv.pi * y)
        product = iv.mpf(1)
        series = iv.mpf(0)
        power = q
        for n in range(1, terms + 1):
            factor = 1 - power
            product *= factor
            series += n * power / factor
            power *= q
        product *= 1 - unit_interval * power / (1 - q)
        series += unit_interval * power * (terms + 1 - terms * q) / (1 - q)**3
        value = product * iv.exp(-iv.pi * y / 12)
        slope = iv.pi * y * (2 * series - iv.mpf(1) / 12)
        if log_y < 0:
            value *= iv.exp(box(-log_y / 2))
            slope = -iv.mpf(1) / 2 - slope
        return value, slope

    weights = interpolation_weights()
    node_count = end * resolution * degree + 1
    node_weights = [Fraction(0)] * node_count
    for start in range(0, node_count - 1, degree):
        for index, weight in enumerate(weights):
            node_weights[start + index] += weight / resolution

    values = [iv.mpf(0) for _ in range(5)]
    for i, weight in enumerate(node_weights):
        u = Fraction(i, resolution * degree)
        u_iv = box(u)
        plus, plus_slope = eta_and_log_derivative(u + scale)
        minus, minus_slope = eta_and_log_derivative(u - scale)
        w = box(weight) * iv.exp(u_iv / 2) * plus * minus
        angle = box(frequency * u)
        cosine = iv.cos(angle)
        sine = iv.sin(angle)
        wa = w * (plus_slope - minus_slope)
        values[0] += w * cosine            # H
        values[1] -= w * u_iv * sine       # H_x
        values[2] += wa * cosine           # H_a
        values[3] -= wa * u_iv * sine      # H_ax
        values[4] -= w * u_iv**2 * cosine  # H_xx

    analytic_error = end * 10**7 * Fraction(1, 8)**(degree + 1) + Fraction(1, 10**80)
    h, hx, ha, hax, hxx = [v + symmetric_interval * box(analytic_error) for v in values]
    jacobian = [(ha, hx), (hax, hxx)]
    rational_inverse = [[Fraction(entry) for entry in row] for row in inverse_text]
    assert (rational_inverse[0][0] * rational_inverse[1][1]
            - rational_inverse[0][1] * rational_inverse[1][0]) != 0
    inverse = [[box(entry) for entry in row] for row in rational_inverse]

    residual = norm([sum(coefficient * v for coefficient, v in zip(row, (h, hx)))
                     for row in inverse])
    inverse_error = matrix_norm([
        [int(i == j) - sum(inverse[i][k] * jacobian[k][j] for k in range(2))
         for j in range(2)]
        for i in range(2)
    ])
    derivative_change = box(50_000 * radius)
    contraction = inverse_error + matrix_norm(inverse) * derivative_change

    # Banach's theorem on the closed box, with all comparisons in intervals.
    assert contraction.b < 1
    assert (residual + contraction * box(radius)).b < box(radius).a
    ha_box = ha + symmetric_interval * derivative_change
    hxx_box = hxx + symmetric_interval * derivative_change
    assert ha_box.b < 0 and hxx_box.a > 0
    scale_interval = box(scale) + symmetric_interval * box(radius)
    frequency_interval = box(frequency) + symmetric_interval * box(radius)
    print("Çift sıfır kutusu (a, x)")
    print(scale_interval)
    print(frequency_interval)
    print("||M F(merkez)|| üst sınırı", residual)
    print("Daralma katsayısının üst sınırı", contraction.b)
    print("Kutudaki H_a", ha_box)
    print("Kutudaki H_xx", hxx_box)
    return {
        "center": {"scale": scale_text, "frequency": frequency_text},
        "box_radius": radius,
        "scale_interval": scale_interval,
        "frequency_interval": frequency_interval,
        "inverse_jacobian_approximation": rational_inverse,
        "center_displacement_upper_bound": residual,
        "contraction_factor_upper_bound": contraction.b,
        "H_a_on_box": ha_box,
        "H_xx_on_box": hxx_box,
        "analytic_error_bound_per_integral": analytic_error,
        "interval_precision_digits": iv.dps,
    }


collision_certificate = certify_collision(
    '1.45200626937306544414442672465215400624761',
    '14.8380401130710086477956510630327906219061',
    [[-387250867, 0], [-948213121, 1602446079]],
)


## Asal sayımı ile kalibrasyon

### $t^3\equiv 2\pmod p$ deneyi

Bu defter $p>3$ asalları için $t^3\equiv2\pmod p$ denkleminin kök sayısı $r_p$'yi hesaplar ve

$$D(x)=\sum_{3<p\le x}(r_p-1)\log p$$

toplamını inceler. $p\equiv 2\pmod3$ için küp alma bir permütasyondur ve $r_p=1$. $p\equiv 1\pmod3$ için $2^{(p-1)/3}\equiv1\pmod p$ ise $r_p=3$, aksi halde $r_p=0$.

Colab'da CPU ile çalıştırın. `LIMIT` varsayılan olarak $10^7$. Daha geniş bir tarama için önce $10^8$, ardından istenirse $10^9$ denenebilir. Defter çalışma süresini ekrana yazar. Kök sayıları ve sınıf adetleri tam tamsayı hesabıdır. Logaritmalı toplam ve grafik kayan nokta hesabı kullanır. Grafikteki $D(x)/\sqrt{x}$ değerleri seçilen örnek noktalarını gösterir. Sonlu bir tarama, tüm $x$ için bir hata sınırını veya Riemann hipotezini kanıtlamaz.

Bu bölüm karşı örneğin ispatında kullanılmaz. Asal toplamının sonlu ölçeklerdeki davranışını gösterir.

In [ ]:
# @title Hesaplama ayarları
LIMIT = 10_000_000 # @param {type:"integer"}
ODDS_PER_SEGMENT = 1_000_000
POINTS_PER_DECADE = 30


In [ ]:
from math import isqrt, log, log10, sqrt
from time import perf_counter


def base_odd_primes(limit):
    """Tek sayı 2*i+1, i indisinde saklanır."""
    flags = bytearray(b"\x01") * ((limit + 1) // 2)
    if flags:
        flags[0] = 0
    for prime in range(3, isqrt(limit) + 1, 2):
        if flags[prime // 2]:
            start = prime * prime // 2
            count = (len(flags) - 1 - start) // prime + 1
            flags[start::prime] = b"\x00" * count
    return [2 * i + 1 for i, is_prime in enumerate(flags) if is_prime]


def primes_after_three(limit, odds_per_segment):
    """5'ten başlayıp üst sınırı da kapsayan bölümlü Eratosthenes eleği."""
    base_primes = base_odd_primes(isqrt(limit))
    last_odd = limit if limit % 2 else limit - 1
    for lo in range(5, last_odd + 1, 2 * odds_per_segment):
        hi = min(last_odd, lo + 2 * (odds_per_segment - 1))
        size = (hi - lo) // 2 + 1
        flags = bytearray(b"\x01") * size
        for prime in base_primes:
            if prime * prime > hi:
                break
            first = max(prime * prime, ((lo + prime - 1) // prime) * prime)
            if first % 2 == 0:
                first += prime
            if first <= hi:
                start = (first - lo) // 2
                count = (size - 1 - start) // prime + 1
                flags[start::prime] = b"\x00" * count

        index = flags.find(b"\x01")
        while index != -1:
            yield lo + 2 * index
            index = flags.find(b"\x01", index + 1)


def root_count(prime):
    """Asal prime > 3 için t³ = 2 (mod prime) denkleminin kök sayısı."""
    if prime % 3 == 2:
        return 1
    return 3 if pow(2, (prime - 1) // 3, prime) == 1 else 0


def check_small_primes():
    trial = [p for p in range(5, 201)
             if all(p % d for d in range(2, isqrt(p) + 1))]
    # Segment sınırlarını da sınamak için testte küçük segmentler kullanılır.
    assert list(primes_after_three(200, 7)) == trial
    assert list(primes_after_three(199, 7)) == trial
    for prime in trial:
        direct = sum((t**3 - 2) % prime == 0 for t in range(prime))
        assert root_count(prime) == direct, (prime, direct)
    assert root_count(7) == 0 and root_count(31) == 3
    return len(trial)


small_primes_checked = check_small_primes()
print(f"Kontrol geçti. 200'e kadar {small_primes_checked} asal tek tek doğrulandı.")


In [ ]:
def compute_discrepancy(limit, odds_per_segment, points_per_decade):
    if limit < 10 or odds_per_segment < 1 or points_per_decade < 1:
        raise ValueError("LIMIT ≥ 10, segment ve nokta sayısı pozitif olmalı.")

    powers_of_ten = {10**k for k in range(1, int(log10(limit)) + 1) if 10**k <= limit}
    log_steps = (
        round(10 ** (k / points_per_decade))
        for k in range(points_per_decade, int(log10(limit) * points_per_decade) + 1)
    )
    checkpoints = sorted(powers_of_ten | {x for x in log_steps if 10 <= x <= limit} | {limit})
    rows = []
    next_point = 0
    counts = [0, 0, 0, 0]
    total = 0.0
    correction = 0.0

    for prime in primes_after_three(limit, odds_per_segment):
        while next_point < len(checkpoints) and checkpoints[next_point] < prime:
            x = checkpoints[next_point]
            rows.append((x, counts[0], counts[1], counts[3], total, total / sqrt(x)))
            next_point += 1

        roots = root_count(prime)
        counts[roots] += 1
        if roots != 1:
            # Kahan toplamı, kayan nokta yuvarlama hatasını azaltır.
            summand = (roots - 1) * log(prime) - correction
            updated = total + summand
            correction = (updated - total) - summand
            total = updated

    for x in checkpoints[next_point:]:
        rows.append((x, counts[0], counts[1], counts[3], total, total / sqrt(x)))

    return rows, powers_of_ten


scan_settings = {
    "limit": LIMIT,
    "odd_integers_per_segment": ODDS_PER_SEGMENT,
    "points_per_decade": POINTS_PER_DECADE,
}
start = perf_counter()
rows, decade_points = compute_discrepancy(LIMIT, ODDS_PER_SEGMENT, POINTS_PER_DECADE)
elapsed = perf_counter() - start

print(f"{LIMIT:,}'e kadar tarama {elapsed:,.2f} saniye sürdü.")
print(f"{sum(rows[-1][1:4]):,} asal sınıflandırıldı (2 ve 3 hariç).")
print()
print(f"{'x':>14} {'0 kök':>12} {'1 kök':>12} {'3 kök':>12} {'D(x)':>14} {'D(x)/√x':>12}")
for x, roots_0, roots_1, roots_3, total, scaled in rows:
    if x in decade_points or x == LIMIT:
        print(f"{x:14,} {roots_0:12,} {roots_1:12,} {roots_3:12,} {total:14,.3f} {scaled:12.5f}")


In [ ]:
import plotly.graph_objects as go

figure = go.Figure(go.Scatter(
    x=[row[0] for row in rows],
    y=[row[5] for row in rows],
    mode="lines+markers",
    marker={"size": 4},
    line={"color": "#19588f", "width": 2},
    hovertemplate="x = %{x:,}<br>D(x)/√x = %{y:.5f}<extra></extra>",
))
figure.add_hline(y=0, line={"color": "#77808b", "width": 1, "dash": "dot"})
figure.update_layout(
    title="Karekök ölçeğinde asal toplamı",
    xaxis={"type": "log", "title": "x"},
    yaxis_title="D(x) / √x",
    template="plotly_white",
)
figure.show()


## Sonuçları ZIP olarak indir

Colab'da **Çalışma zamanı → Tümünü çalıştır** seçildiğinde bu hücre `kubik_L_sonuclari.zip` indirmesini başlatır. Arşivde `sonuclar.json` bulunur. Dosya, Laguerre hesabının, reel olmayan sıfır disklerinin, çift sıfır kutusunun ve asal taramasındaki tüm örneklerin sonuçlarını içerir. Aralık uçları kesin ondalık dizeler, rasyonel parametreler ise kesir dizeleri olarak saklanır. Asal adetleri tamsayı, logaritmalı toplamlar kayan noktalı JSON sayılarıdır.

In [ ]:
# @title Sonuçları JSON olarak paketle ve ZIP indir
import json
from pathlib import Path
from platform import python_version
from zipfile import ZIP_DEFLATED, ZipFile

import mpmath


def decimal_endpoint(endpoint):
    # mpmath bir aralığın uçlarını işaret, mantis ve 2'nin üssüyle saklar.
    # Bu bilgi tam ondalık yazıma çevrilir, kullanıcıya gösterilen yuvarlak çıktıya değil.
    sign, mantissa, exponent, bit_count = endpoint
    if bit_count < 0:
        raise ValueError("Aralık ucu sonlu olmalı.")
    if not mantissa:
        return "0"
    if exponent >= 0:
        digits = str(mantissa << exponent)
    else:
        places = -exponent
        digits = str(mantissa * 5**places).zfill(places + 1)
        digits = digits[:-places] + "." + digits[-places:]
    return "-" + digits if sign else digits


def encode_exact(value):
    if isinstance(value, Fraction):
        return str(value)
    if isinstance(value, type(iv.mpf(0))):
        lower, upper = value._mpi_
        return {"lower": decimal_endpoint(lower), "upper": decimal_endpoint(upper)}
    raise TypeError(f"JSON'a aktarılamayan tür {type(value).__name__}")


report = {
    "schema_version": 1,
    "software": {"python": python_version(), "mpmath": mpmath.__version__},
    "laguerre_counterexample": {
        "scale": scale,
        "frequency": frequency,
        "H": values[0],
        "H_x": values[1],
        "H_xx": values[2],
        "laguerre_expression": laguerre,
        "analytic_error_bound_per_integral": error,
        "interval_precision_digits": 45,
    },
    "nonreal_zero_pair": nonreal_certificate,
    "double_zero": collision_certificate,
    "prime_scan": {
        **scan_settings,
        "excluded_primes": [2, 3],
        "small_primes_checked_to_200": small_primes_checked,
        "prime_count": sum(rows[-1][1:4]),
        "elapsed_seconds": elapsed,
        "samples": [
            {"x": x, "zero_roots": roots_0, "one_root": roots_1, "three_roots": roots_3,
             "D": total, "D_over_sqrt_x": scaled}
            for x, roots_0, roots_1, roots_3, total, scaled in rows
        ],
    },
}

archive_path = Path("kubik_L_sonuclari.zip").resolve()
payload = json.dumps(report, ensure_ascii=False, allow_nan=False, indent=2, default=encode_exact)
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    archive.writestr("sonuclar.json", payload.encode("utf-8"))
with ZipFile(archive_path) as archive:
    if archive.testzip() is not None:
        raise OSError("ZIP bütünlük denetiminden geçemedi.")

print(f"Hazır: {archive_path.name} ({len(rows)} asal taraması noktası).")
try:
    from google.colab import files
except ImportError:
    print(archive_path)
else:
    files.download(str(archive_path))
